# Cinema Sales Analysis

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)


def section(title):
    """Print a clearly separated section header."""
    print("\n" + "=" * 70)
    print(f"  {title}")
    print("=" * 70 + "\n")

## 1. Load and inspect

In [ ]:
section("1. LOAD AND INSPECT")
df = pd.read_csv("15_cinema_sales.csv")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns \n")
print("Column types:\n")
print(df.dtypes)
print("\nFirst 5 rows:\n")
print(df.head())
print("\nSummary statistics:\n")
print(df.describe(include="all"))

## 2. Missing values and duplicate Transaction_IDs

In [ ]:
section("2. MISSING VALUES AND DUPLICATES")
print("Missing values per column:\n")
print(df.isna().sum())
dupes = df["Transaction_ID"].duplicated().sum()
print(f"\nDuplicate Transaction_IDs: {dupes}")
df = df.drop_duplicates(subset="Transaction_ID", keep="first")
print(f"Rows after removing duplicates: {len(df)}")

## 3. Clean Concession_Spend

In [ ]:
# Missing spend most plausibly means "bought nothing" if the till only logs purchases.
section("3. CLEAN CONCESSION_SPEND")
missing_spend = df["Concession_Spend"].isna().sum()
df["Concession_Spend"] = df["Concession_Spend"].fillna(0)
print(f"Filled {missing_spend} missing Concession_Spend values with 0")

## 4. Dates and weekend flag

In [ ]:
section("4. DATES AND WEEKEND FLAG")
df["Show_Date"] = pd.to_datetime(df["Show_Date"], errors="coerce")
bad_dates = df["Show_Date"].isna().sum()
print(f"Unparseable dates dropped: {bad_dates}")
df = df.dropna(subset=["Show_Date"])
df["Weekday"] = df["Show_Date"].dt.day_name()
df["Is_Weekend"] = np.where(df["Weekday"].isin(["Saturday", "Sunday"]), "Weekend", "Weekday")
print("\nTransactions by day type:\n")
print(df["Is_Weekend"].value_counts())

## 5. Revenue totals

In [ ]:
# Adjust if Ticket_Revenue isn't a column
section("5. REVENUE TOTALS")
if "Ticket_Revenue" not in df:
    df["Ticket_Revenue"] = df["Ticket_Quantity"] * df["Ticket_Price"]
df["Total_Spend"] = df["Ticket_Revenue"] + df["Concession_Spend"]
print(df[["Ticket_Revenue", "Concession_Spend", "Total_Spend"]].sum().round(2))

## 6. Top 10 movies by tickets sold

In [ ]:
section("6. TOP 10 MOVIES BY TICKETS SOLD")
top10 = (df.groupby("Movie_Title")
           .agg(Tickets=("Ticket_Quantity", "sum"), Revenue=("Ticket_Revenue", "sum"))
           .sort_values("Tickets", ascending=False).head(10))
print(top10.round(2))

## 7. Average occupancy by screen format

In [ ]:
section("7. AVERAGE OCCUPANCY BY SCREEN FORMAT")
print(df.groupby("Screen_Format")["Occupancy_Pct"].mean().sort_values(ascending=False).round(2))

## 8. Weekday vs weekend

In [ ]:
section("8. WEEKDAY VS WEEKEND")
print(df.groupby("Is_Weekend")[["Ticket_Quantity", "Ticket_Revenue", "Concession_Spend"]]
        .agg(["sum", "mean"]).round(2))
days = df.groupby("Is_Weekend")["Show_Date"].nunique()
print("\nDistinct dates per group:\n")
print(days)
print("\nAverage revenue per day:\n")
print((df.groupby("Is_Weekend")["Ticket_Revenue"].sum() / days).round(2))

## 9. Showtime slots

In [ ]:
section("9. SHOWTIME SLOTS")
print(df.groupby("Showtime_Slot")
        .agg(Tickets=("Ticket_Quantity", "sum"),
             Transactions=("Transaction_ID", "count"),
             Avg_Occupancy=("Occupancy_Pct", "mean")).round(2))

## 10. Concession attach rate

In [ ]:
section("10. CONCESSION ATTACH RATE")
attach = (df["Concession_Spend"] > 0).mean() * 100
print(f"Overall attach rate: {attach:.1f}%\n")
print("By day type (%):\n")
print(df.groupby("Is_Weekend")["Concession_Spend"].apply(lambda s: (s > 0).mean() * 100).round(1))

## 11. Average spend by genre

In [ ]:
section("11. AVERAGE SPEND BY GENRE")
print(df.groupby("Genre")[["Ticket_Revenue", "Concession_Spend"]].mean().round(2))

## 12. Total spend statistics (NumPy)

In [ ]:
section("12. TOTAL SPEND STATISTICS")
spend = df["Total_Spend"].dropna().to_numpy()
print(f"Mean   : {np.mean(spend):.2f}")
print(f"Median : {np.median(spend):.2f}")
print(f"Std Dev: {np.std(spend, ddof=1):.2f}")

## 13. Ticket quantity outliers (IQR)

In [ ]:
section("13. TICKET QUANTITY OUTLIERS (IQR)")
q1, q3 = np.percentile(df["Ticket_Quantity"], [25, 75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
large = df[df["Ticket_Quantity"] > upper]
print(f"Q1: {q1}  Q3: {q3}  IQR: {iqr}")
print(f"Upper fence: {upper}")
print(f"Large bookings: {len(large)}")
print("\nTicket quantity distribution:\n")
print(df["Ticket_Quantity"].value_counts().sort_index())

## 14. Group-size bands

In [ ]:
section("14. GROUP-SIZE BANDS")
df["Group_Band"] = pd.cut(df["Ticket_Quantity"], bins=[0, 1, 2, 5, np.inf],
                          labels=["Solo", "Couple", "Small Group", "Large Group"])
print(df.groupby("Group_Band", observed=True)
        .agg(Transactions=("Transaction_ID", "count"),
             Avg_Concession=("Concession_Spend", "mean")).round(2))

print("\n" + "=" * 70)
print("  ANALYSIS COMPLETE")
print("=" * 70 + "\n")